In [1]:
# here we are implementing trimming of messages in short term memory

In [13]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from psycopg_pool import ConnectionPool
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_core.messages import HumanMessage
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
import os

In [3]:
load_dotenv()

True

In [4]:
LLM = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [6]:
DB_URI = os.environ["DATABASE_URL"]

connection_kwargs = {
    "autocommit":True,
    "prepare_threshold":0
}

In [9]:
pool = ConnectionPool(
    conninfo=DB_URI,
    max_size=20,
    kwargs=connection_kwargs
)

checkpointer = PostgresSaver(pool)
checkpointer.setup()

In [11]:
MAX_TOKENS = 150

def call_model(state:MessagesState):

    # trim conversation history -> last n messages that fit within token limit
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print(f"Current token count -> {count_tokens_approximately(messages=messages)}")

    for message in messages:
        print(message.content)

    response = LLM.invoke(messages)
    return {"messages":[response]}

In [12]:
graph_builder = StateGraph(MessagesState)

graph_builder.add_node("call_model",call_model)

graph_builder.add_edge(START,"call_model")
graph_builder.add_edge("call_model",END)

graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
thread_id = "101"

graph.invoke({"messages":[HumanMessage(content="who is prime minister of america?")]},config={"configurable":{"thread_id":thread_id}})

Current token count -> 13
who is prime minister of america?


{'messages': [HumanMessage(content='i am bob', additional_kwargs={}, response_metadata={}, id='b098a9d5-830a-426a-a639-3d021d5e28b2'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 38, 'total_tokens': 64, 'completion_time': 0.062145885, 'completion_tokens_details': None, 'prompt_time': 0.005413971, 'prompt_tokens_details': None, 'queue_time': 0.051172489, 'total_time': 0.067559856}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdb90-b248-7ba0-992f-6c033e3d4535-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 26, 'total_tokens': 64}),
  HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id

In [17]:
graph.invoke({"messages":[HumanMessage(content="ok then who is president of america?")]},config={"configurable":{"thread_id":thread_id}})

Current token count -> 13
ok then who is president of america?


{'messages': [HumanMessage(content='i am bob', additional_kwargs={}, response_metadata={}, id='b098a9d5-830a-426a-a639-3d021d5e28b2'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 38, 'total_tokens': 64, 'completion_time': 0.062145885, 'completion_tokens_details': None, 'prompt_time': 0.005413971, 'prompt_tokens_details': None, 'queue_time': 0.051172489, 'total_time': 0.067559856}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdb90-b248-7ba0-992f-6c033e3d4535-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 26, 'total_tokens': 64}),
  HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id

In [18]:
response = graph.invoke({"messages":[HumanMessage(content="can u tell me who is he?")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 86
ok then who is president of america?
As of my knowledge cutoff in 2023, the President of the United States is Joe Biden. However, please note that my information may not be up-to-date, and I recommend checking a reliable news source for the most current information.
can u tell me who is he?
Joe Biden is the 46th President of the United States. Here's a brief overview of who he is:

**Early Life:**
Joe Biden was born on November 20, 1942, in Scranton, Pennsylvania. He grew up in a middle-class family and was the oldest of four siblings. He graduated from the University of Delaware and later earned a law degree from Syracuse University.

**Career:**
Before becoming President, Joe Biden had a long and distinguished career in public service. He was:

1. **Senator:** Biden served six terms as a United States Senator from Delaware, from 1973 to 2009.
2. **Vice President:** He served as Vice President under President Barack Obama from 2009 to 2017.
3. **Chairman of t

In [19]:
response = graph.invoke({"messages":[HumanMessage(content="ohh then what about the current state of america what's happening there?")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 22
ohh then what about the current state of america what's happening there?
The current state of America is complex and multifaceted. As my knowledge cutoff is March 2023, I'll provide an overview of the major events and trends up to that point. Keep in mind that the situation may have evolved since then.

**Politics:**

1. **Biden Administration:** President Joe Biden has been in office since January 2021, focusing on issues like climate change, healthcare, and economic recovery.
2. **Congress:** The 118th Congress has been marked by partisan divisions, with Democrats controlling the Senate and Republicans holding the House of Representatives.
3. **Elections:** The 2022 midterm elections saw a mixed outcome, with Democrats performing better than expected, but Republicans gaining control of the House.

**Social Issues:**

1. **Racial Tensions:** The Black Lives Matter movement and discussions around police brutality, systemic racism, and social justice continue t

In [20]:
response = graph.invoke({"messages":[HumanMessage(content="nice you are really helpful")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 11
nice you are really helpful
I'm glad to hear that I've been able to assist you. My goal is to provide helpful and accurate information, and it's great to know that I've been able to do so. If you have any more questions or need further assistance, feel free to ask!


In [21]:
response = graph.invoke({"messages":[HumanMessage(content="u know what my name is?")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 86
nice you are really helpful
I'm glad to hear that I've been able to assist you. My goal is to provide helpful and accurate information, and it's great to know that I've been able to do so. If you have any more questions or need further assistance, feel free to ask!
u know what my name is?
I don't have any information about your name. I'm a large language model, I don't have the ability to store or recall personal information about individuals, including names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!


In [22]:
response = graph.invoke({"messages":[HumanMessage(content="I am bob")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 121
u know what my name is?
I don't have any information about your name. I'm a large language model, I don't have the ability to store or recall personal information about individuals, including names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!
I am bob
Nice to meet you, Bob! It's great to have a name to associate with our conversation. How's your day going so far, Bob? Is there something on your mind that you'd like to talk about, or are you just looking for some casual chat?


In [23]:
response = graph.invoke({"messages":[HumanMessage(content="just say hello")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 76
I am bob
Nice to meet you, Bob! It's great to have a name to associate with our conversation. How's your day going so far, Bob? Is there something on your mind that you'd like to talk about, or are you just looking for some casual chat?
just say hello
Hello


In [24]:
response = graph.invoke({"messages":[HumanMessage(content="just say hello and my name")]},config={"configurable":{"thread_id":thread_id}})
print(response["messages"][-1].content)

Current token count -> 94
I am bob
Nice to meet you, Bob! It's great to have a name to associate with our conversation. How's your day going so far, Bob? Is there something on your mind that you'd like to talk about, or are you just looking for some casual chat?
just say hello
Hello
just say hello and my name
Hello Bob
